# mini GPT Colab 실행 노트북

이 노트북은 Colab에서 `seonho` 브랜치를 자동으로 clone하고, CUDA/GPU 사용 여부를 확인한 뒤 바로 테스트와 기본 smoke 실행까지 진행할 수 있게 구성했습니다.

In [ ]:
# 1. Colab / CUDA / repository setup
import os
import random
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/seonho12-54/WEEK14_AI.git"
BRANCH = "seonho"
REPO_DIR = Path("/content/WEEK14_AI") if "google.colab" in sys.modules else Path.cwd()

if "google.colab" in sys.modules:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR, check=True)
        subprocess.run(["git", "checkout", BRANCH], cwd=REPO_DIR, check=True)
        subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=REPO_DIR, check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    os.chdir(REPO_DIR)

sys.path.insert(0, str(REPO_DIR / "src"))

import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Repo:", REPO_DIR)
print("Torch:", torch.__version__)
print("Device:", device)
if device.type == "cuda":
    print("CUDA:", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# 2. NSMC data download / preprocessing
import download_data

paths = download_data.main()
paths


In [ ]:
# 3. Run all tests
result = subprocess.run([sys.executable, "-m", "pytest", "tests", "-v"], cwd=REPO_DIR)
assert result.returncode == 0, "pytest failed. Check the output above."


In [ ]:
# 4. One-batch smoke test on CPU/GPU
from bpe import BPETokenizer
from dataset import create_dataloader
from model import GPTModel
from train import calc_loss_batch, generate

corpus_path = REPO_DIR / "data" / "nsmc_lm_train.txt"
corpus = corpus_path.read_text(encoding="utf-8")[:5000] if corpus_path.exists() else "이 영화는 정말 좋았다.\n별로였다."
if not corpus.strip():
    corpus = "이 영화는 정말 좋았다.\n별로였다."

tokenizer = BPETokenizer(vocab_size=300)
tokenizer.train(corpus)
token_ids = tokenizer.encode(corpus, add_bos_eos=True)
while len(token_ids) < 80:
    token_ids.extend(token_ids)

context_length = 32
loader = create_dataloader(token_ids, context_length=context_length, batch_size=2, shuffle=False)
input_batch, target_batch = next(iter(loader))

config = {
    "vocab_size": len(tokenizer.id_to_token),
    "context_length": context_length,
    "emb_dim": 64,
    "n_heads": 4,
    "n_layers": 1,
    "drop_rate": 0.0,
    "qkv_bias": False,
}
model = GPTModel(config).to(device)
loss = calc_loss_batch(input_batch, target_batch, model, device)
loss.backward()
print("smoke loss:", float(loss.item()))

start = torch.tensor([tokenizer.encode("이 영화는", add_bos_eos=True)], dtype=torch.long, device=device)
generated = generate(model, start, max_new_tokens=10, context_size=context_length, temperature=1.0, top_k=20, eos_id=tokenizer.get_eos_id())
print(tokenizer.decode(generated[0].tolist(), skip_special=True))
